# SQL Practice #16: Second Most Recent Activity per User

**Question:** Given a table of user activities with start and end dates, find the **second most recent activity** for each user. If a user has only one activity, return that single activity.

**Approach:**
1. Create the `UserActivity` table and insert sample data (Cell 1–2)
2. Solve with SQL using `ROW_NUMBER()` and `COUNT()` window functions (Cell 3)
3. Reproduce the same solution in PySpark (Cell 4–6)

In [0]:
%sql
-- ============================================================
-- Cell 1: Create the UserActivity table and insert sample data
-- ============================================================
create table b_sql.b_practice.UserActivity
(
username      varchar(20) ,
activity      varchar(20),
startDate     Date   ,
endDate      Date
);

insert into b_sql.b_practice.UserActivity values 
('Alice','Travel','2020-02-12','2020-02-20')
,('Alice','Dancing','2020-02-21','2020-02-23')
,('Alice','Travel','2020-02-24','2020-02-28')
,('Bob','Travel','2020-02-11','2020-02-18');

In [0]:
%sql
-- ============================================================
-- Cell 2: Preview the UserActivity table
-- ============================================================
select * from b_sql.b_practice.UserActivity

In [0]:
%sql
-- ============================================================
-- Cell 3: SQL Solution
-- Find the second most recent activity per user.
-- If a user has only one activity, return that activity.
-- ============================================================
with cte as (
select *,row_number() over(partition by username order by startdate desc) rn,
count(*) over(partition by username) cnt
from b_sql.b_practice.UserActivity)
select *
from cte
where rn = 2  or  cnt < 2 AND rn = cnt

In [0]:
# ============================================================
# Cell 4: Load the UserActivity table into a PySpark DataFrame
# ============================================================
df_activity=spark.read.table("b_sql.b_practice.UserActivity")
df_activity.display()

In [0]:
# ============================================================
# Cell 5: Import PySpark functions and Window
# ============================================================
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [0]:

# ============================================================
# Cell 6: PySpark Solution
# Find the second most recent activity per user.
# If a user has only one activity, return that activity.
# (PySpark equivalent of the SQL query in Cell 3)
# ============================================================

# Window spec: rank activities per user by startDate descending (most recent first)
window_spec = Window.partitionBy("username").orderBy(col("startDate").desc())

# Window spec: count total activities per user (no ordering needed for aggregate)
count_spec = Window.partitionBy("username")

# Add row number (rnk) and total count (cnt) per user
df_activity1 = (
    df_activity
    .withColumn("rnk", row_number().over(window_spec))   # 1 = most recent
    .withColumn("cnt", count(lit(1)).over(count_spec))     # total activities per user
)

df_activity1.display()

# Filter: get the 2nd most recent activity (rnk == 2),
# or if the user has only 1 activity (cnt < 2), return that activity (rnk == cnt)
df_activity_final = df_activity1.filter(
    (col("rnk") == 2) |
    ((col("cnt") < 2) & (col("rnk") == col("cnt")))
)

df_activity_final.display()

